# Pipeline continuous integration

#### Explanation
*Detail in report.MD*

1. First Stage - Unzipping
Due to Google Colab environment, either you can connect to Gdrive or upload zip file. Therefore, the code will receive a zip file that contains original folders provided by Solafune and unzip it automatically.

2. Second Stage - Pre-process and Checking Consistency
In order to stack file into RGB file, it must be checked from the properties of tif file, example CRS, Height & Width, Transform and so on.

3. Third Stage - Detection Stage
The function will apply VARI analysis (reference https://docs.geopera.com/spectral-indices/vari_visible and https://pmc.ncbi.nlm.nih.gov/articles/PMC9332278/ NDVI/VARI for open-pit mining) combine with either Median Absolute Detection or Clustering to extract the feature changes.

4. Fourt Pipeline Stage - Convert to geopandas and save to gpkg.
After performing feature extraction in numpy multi-array based, the code will convert into geopandas and save into gpkg file. The code has option to view the gpkg in query style and also reload it back from gpkg to geopandas dataframe.

5. Fifth Pipeline Stage - Visualization
geopandas has explore() function to enable visualizing the map with interactive style and the base-map is used from ESRI arcgis.

6. Last Pipeline Stage - Unite All Functions into a single function
Stack all pipeline stage into a single function.

In [ ]:
#@title 1. Pipeline Functions
import zipfile
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import folium
import matplotlib.pyplot as plt
from IPython.display import display
from rasterio.features import shapes, sieve, rasterize
from shapely.geometry import shape
from sklearn.cluster import KMeans


# ======================= helpers =======================
def ensure_dir(folder):
    """Make sure `folder` exists as a real directory. Create it if missing."""
    folder = Path(folder)
    if folder.is_dir():
        return folder
    if folder.is_symlink() and not folder.exists():       # broken symlink
        folder.unlink()
    elif folder.exists():                                 # a file with this name
        raise NotADirectoryError(f"'{folder}' exists but is a file, not a folder.")
    try:
        folder.mkdir(parents=True, exist_ok=True)
    except FileExistsError:
        if not folder.is_dir():
            raise
    return folder


def band_paths(folder, bands):
    """Paths of the band files, in the given order. Only these files are used (stacks are never picked up)."""
    paths = [Path(folder) / f"{b}.tif" for b in bands]
    missing = [p.name for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"{folder}: missing band files {missing}")
    return paths


# ======================= step 1: unzip =======================
def unzip_data(zip_path, unzip_to):
    """ User could put Zip file to Google Colab and checklist the unzip file"""
    ensure_dir(unzip_to)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(unzip_to)
    print(f"Unzipped {zip_path} -> {unzip_to}")


# ======================= step 2: checks, stacks, previews =======================
def check_inputs(folders, bands, show=True):
    """Check that every band file shares one grid, and compare band statistics between dates."""
    rows = []
    for folder in folders:
        for p in band_paths(folder, bands):
            with rasterio.open(p) as src:
                t = src.transform
                a = src.read(1).astype("float64")
                grid = dict(width=src.width, height=src.height, crs=str(src.crs),
                            res_x=t.a, res_y=t.e, origin_x=t.c, origin_y=t.f)
            v = a[a != 0]                                   # zero = nodata
            rows.append({"folder": Path(folder).name, "band": p.stem, **grid,
                         "min": v.min(), "max": v.max(), "mean": v.mean(), "std": v.std(),
                         "p2": np.percentile(v, 2), "p98": np.percentile(v, 98)})
    df = pd.DataFrame(rows)

    grid_cols = ["width", "height", "crs", "res_x", "res_y", "origin_x", "origin_y"]
    ref = df.iloc[0][grid_cols]
    df["grid_match"] = (df[grid_cols] == ref).all(axis=1)

    if show:
        pd.set_option("display.width", 250, "display.max_columns", None)
        display(df.round(2))
        mean_tbl = df.pivot(index="band", columns="folder", values="mean")
        first, last = df["folder"].iloc[0], df["folder"].iloc[-1]
        mean_tbl["ratio (last / first)"] = mean_tbl[last] / mean_tbl[first]
        display(mean_tbl.round(3))

    if not df["grid_match"].all():
        display(df.loc[~df["grid_match"], ["folder", "band"] + grid_cols])
        raise ValueError("Band files do not share the same grid (see table above).")
    print("✅ All band files share identical CRS, transform and dimensions.")
    return df


def stack_bands(folder, bands, out_dir):
    """Write one multi-band tif per date: band 1 = blue, 2 = green, 3 = red."""
    paths = band_paths(folder, bands)
    out_file = ensure_dir(out_dir) / f"{Path(folder).name}_stack.tif"
    with rasterio.open(paths[0]) as src:
        meta = src.meta.copy()
    meta.update(count=len(paths))
    with rasterio.open(out_file, "w", **meta) as dst:
        for i, p in enumerate(paths, start=1):
            with rasterio.open(p) as src:
                dst.write(src.read(1), i)
            dst.set_band_description(i, p.stem)
    print(f"Stack saved: {out_file}")
    return out_file


def rgb_preview(folder, bands, out_png):
    """True-colour PNG with a 2-98% stretch per band. `bands` must be (blue, green, red)."""
    arrays = []
    for p in band_paths(folder, bands):
        with rasterio.open(p) as src:
            a = src.read(1).astype("float32")
        a[a == 0] = np.nan
        lo, hi = np.nanpercentile(a, [2, 98])
        arrays.append(np.nan_to_num(np.clip((a - lo) / (hi - lo), 0, 1)))
    rgb = np.dstack(arrays[::-1])                           # -> red, green, blue
    ensure_dir(Path(out_png).parent)
    plt.imsave(out_png, rgb)
    print(f"Preview saved: {out_png}")
    return rgb


# ======================= step 3: change detection =======================
def classify_mad(diff, K=3):
    """Median Absolute Deviation threshold. Returns (change, None): MAD has no probabilities."""
    mad = np.nanmedian(np.abs(diff - np.nanmedian(diff)))
    threshold = K * 1.4826 * mad
    print(f"[MAD] Threshold (+/-): {threshold:.4f}")
    change = np.zeros(diff.shape, dtype="uint8")
    change[diff < -threshold] = 1
    change[diff > threshold] = 2
    return change, None

def classify_kmeans(diff, min_delta=0.1, sample_size=200_000):
    """K-means (k=3) on the 1-D VARI difference: loss / no change / gain.
    Returns (change, membership). Membership is a distance-based score, NOT a probability."""
    valid = ~np.isnan(diff)
    X = diff[valid].reshape(-1, 1)

    rng = np.random.default_rng(0)
    sample = X[rng.choice(len(X), size=min(sample_size, len(X)), replace=False)]

    # Start from the tails and the middle so the three centres cannot all land in the no-change spike
    init = np.percentile(sample, [2, 50, 98]).reshape(-1, 1)
    km = KMeans(n_clusters=3, init=init, n_init=1, random_state=0).fit(sample)

    centres = km.cluster_centers_.ravel()
    loss_i, none_i, gain_i = np.argsort(centres)
    labels = km.predict(X)

    for name, i in [("loss", loss_i), ("no change", none_i), ("gain", gain_i)]:
        members = X[labels == i, 0]
        std = members.std() if len(members) else np.nan
        print(f"[KMeans] {name:9s} centre={centres[i]:+.3f}  std={std:.3f}  "
              f"share={len(members) / len(X):.1%}")

    # Soft score from distances (inverse squared distance, normalised to sum to 1 per pixel)
    d2 = (X - centres[None, :]) ** 2
    w = 1.0 / np.maximum(d2, 1e-12)
    member = (w / w.sum(axis=1, keepdims=True)).astype("float32")

    change_valid = np.zeros(len(X), dtype="uint8")
    change_valid[labels == loss_i] = 1
    change_valid[labels == gain_i] = 2
    change = np.zeros(diff.shape, dtype="uint8")
    change[valid] = change_valid
    change[np.abs(diff) < min_delta] = 0 # Min delta is used to "classify" no change threshold

    p_loss = np.full(diff.shape, np.nan, dtype="float32")
    p_gain = np.full(diff.shape, np.nan, dtype="float32")
    p_loss[valid] = member[:, loss_i]
    p_gain[valid] = member[:, gain_i]
    return change, np.stack([p_loss, p_gain])


def detect_changes(folder_1, folder_2, bands, boa_offset, method, K, min_delta,
                   output_dir, change_map_name, change_binary_name, show_plots=True):
    method = method.strip().lower()
    if method not in ("mad", "kmeans"):
      raise ValueError("detection_method must be 'MAD' or 'KMeans'")
    def read_band(folder, name):
        with rasterio.open(Path(folder) / f"{name}.tif") as src:
            data = src.read(1).astype("float32")
        data[data == 0] = np.nan
        return np.clip(data - boa_offset, 0, None) / 10000

    def vari(folder):
        """VARI = (Green - Red) / (Green + Red - Blue)"""
        blue, green, red = (read_band(folder, b) for b in bands)
        denom = green + red - blue
        denom[np.abs(denom) < 1e-4] = np.nan
        return np.clip((green - red) / denom, -1, 1)

    vari_1, vari_2 = vari(folder_1), vari(folder_2)
    diff = vari_2 - vari_1
    bias = np.nanmedian(diff)                              # remove the overall shift (haze, sun angle)
    diff = diff - bias

    method = method.strip().lower()

    if method == "mad":
        change, proba = classify_mad(diff, K=K)
    else:
        change, proba = classify_kmeans(diff, min_delta=min_delta)
    change[np.isnan(diff)] = 0

    # Summary
    with rasterio.open(Path(folder_1) / f"{bands[0]}.tif") as src:
        profile = src.profile
        pixel_area_ha = abs(src.transform.a * src.transform.e) / 10_000
    valid = ~np.isnan(diff)
    print(f"Method               : {method.upper() if method == 'mad' else 'Clustering'}")
    print(f"Median shift removed : {bias:.4f}")
    print(f"Valid pixels         : {valid.sum():,}")
    for code, label in [(1, "loss"), (2, "gain")]:
        n = (change == code).sum()
        print(f"Vegetation {label:5s}     : {n * pixel_area_ha:,.1f} ha ({100 * n / valid.sum():.2f}%)")

    if show_plots:
        fig, axes = plt.subplots(1, 4, figsize=(22, 6))
        axes[0].imshow(vari_1, cmap="RdYlGn", vmin=-0.5, vmax=0.5); axes[0].set_title("VARI date 1")
        axes[1].imshow(vari_2, cmap="RdYlGn", vmin=-0.5, vmax=0.5); axes[1].set_title("VARI date 2")
        axes[2].imshow(diff, cmap="RdBu", vmin=-0.3, vmax=0.3);     axes[2].set_title("VARI difference (date 2 - date 1)")
        axes[3].imshow(change, cmap=plt.matplotlib.colors.ListedColormap(["lightgray", "red", "green"]))
        axes[3].set_title("Change map (red = loss, green = gain)")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout(); plt.show()

    # Save rasters on the same grid as the input bands
    out = ensure_dir(output_dir)

    p = profile.copy(); p.update(dtype="float32", count=1, nodata=np.nan, compress="deflate")
    with rasterio.open(out / change_map_name, "w", **p) as dst:        # change intensity
        dst.write(np.abs(diff).astype("float32"), 1)
        dst.set_band_description(1, "abs_vari_difference")

    binary = (change > 0).astype("uint8")
    binary[np.isnan(diff)] = 255
    p = profile.copy(); p.update(dtype="uint8", count=1, nodata=255, compress="deflate")
    with rasterio.open(out / change_binary_name, "w", **p) as dst:     # change vs no change
        dst.write(binary, 1)
        dst.set_band_description(1, "change_binary")

    if proba is not None:
        proba_file = out / (Path(change_map_name).stem + "_proba.tif")
        p = profile.copy(); p.update(dtype="float32", count=2, nodata=np.nan, compress="deflate")
        with rasterio.open(proba_file, "w", **p) as dst:
            dst.write(proba)
            dst.set_band_description(1, "p_loss")
            dst.set_band_description(2, "p_gain")

    print(f"Saved rasters in: {out}")
    return change, proba


# ======================= step 4: polygons =======================
def extract_polygons(folder_1, band, change, min_pixels, proba=None, show_plots=True):
    with rasterio.open(Path(folder_1) / f"{band}.tif") as src:
        transform, crs = src.transform, src.crs
    if crs is not None and crs.is_geographic:
        print("⚠️ CRS is geographic (degrees): area_ha will be wrong. Reproject the bands to a metre-based CRS first.")

    clean = sieve(change, size=min_pixels, connectivity=8)            # remove tiny speckle clusters
    results = shapes(clean, mask=(clean > 0), transform=transform, connectivity=8)
    records = [{"geometry": shape(geom), "class_id": int(value)} for geom, value in results]
    if not records:
        raise ValueError("No change polygons found. Lower K / min_pixels, or switch the detection method.")

    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs=crs)
    gdf["change_type"] = gdf["class_id"].map({1: "loss", 2: "gain"})
    gdf["area_ha"] = gdf.geometry.area / 10_000

    if proba is not None:                                             # confidence (clustering only)
        p_class = np.full(clean.shape, np.nan, dtype="float32")
        p_class[clean == 1] = proba[0][clean == 1]
        p_class[clean == 2] = proba[1][clean == 2]
        poly_id = rasterize(zip(gdf.geometry, gdf.index + 1), out_shape=clean.shape,
                            transform=transform, fill=0, dtype="int32")
        stats = (pd.DataFrame({"id": poly_id.ravel(), "p": p_class.ravel()})
                   .query("id > 0").groupby("id")["p"].agg(["mean", "min"]))
        gdf["conf_mean"] = stats["mean"].reindex(gdf.index + 1).values
        gdf["conf_min"] = stats["min"].reindex(gdf.index + 1).values
        gdf["confidence_class"] = pd.cut(gdf["conf_mean"], bins=[0, 0.7, 0.9, 1.0],
                                         labels=["low", "medium", "high"], include_lowest=True)
        display(gdf.groupby(["change_type", "confidence_class"], observed=True)["area_ha"]
                 .agg(["count", "sum"]).round(2).rename(columns = {'count':"count_total_area","sum":"sum_area_ha"}))

    display(f"Polygons: {len(gdf):,}")
    display(gdf.groupby("change_type")["area_ha"].agg(["count", "sum", "max"]).round(2).rename(columns = {'count':"count_total_area",
                                                                                                          "sum":"sum_area_ha",
                                                                                                           "max":"max_area_ha"}))
    if show_plots:
        gdf.plot(column="change_type", cmap="RdYlGn_r", figsize=(8, 8), legend=True)
        if proba is not None:
            gdf.plot(column="conf_mean", cmap="viridis", figsize=(8, 8), legend=True, vmin=0.5, vmax=1)
        plt.show()
    return gdf


# ======================= step 5: SQLite =======================
def gdf_to_gpkg(gdf: gpd.GeoDataFrame,
                gpkg_path,
                layer: str = "change_features",
                date_before: str | None = None,
                date_after: str | None = None,
                reload: bool = False) -> gpd.GeoDataFrame:
    """Write change polygons to a GeoPackage layer (SQLite file with a true geometry column)."""
    if gdf.crs is None:
        raise ValueError("GeoDataFrame has no CRS; cannot store it in a GeoPackage.")
    if gdf.crs.is_geographic:
        print("⚠️ CRS is geographic (degrees): area_m2 will be wrong. Reproject to a metre-based CRS first.")

    gpkg_path = Path(gpkg_path)
    ensure_dir(gpkg_path.parent)

    out = gdf.copy()
    out["area_m2"] = out.geometry.area
    if date_before is not None:
        out["date_before"] = date_before
    if date_after is not None:
        out["date_after"] = date_after
    if "confidence" not in out.columns:                      # MAD has no probabilities -> stays NaN
        out["confidence"] = out["conf_mean"] if "conf_mean" in out.columns else np.nan
    for c in out.select_dtypes("category").columns:          # categories do not always survive the write
        out[c] = out[c].astype(object)

    out.index = pd.RangeIndex(1, len(out) + 1, name="id")    # stable 1-based id column
    out = out.reset_index()

    out.to_file(gpkg_path, layer=layer, driver="GPKG")       # creates the spatial index automatically

    con = sqlite3.connect(gpkg_path)                         # a GPKG is plain SQLite, so normal indexes work
    try:
        con.execute(f'CREATE INDEX IF NOT EXISTS idx_{layer}_type ON "{layer}"(change_type)')
        con.commit()
    finally:
        con.close()
    print(f"Saved {len(out):,} polygons to {gpkg_path} (layer '{layer}')")

    if reload:
        back = gpkg_to_gdf(gpkg_path, layer)
        print(f"Reload check -> rows: {len(back) == len(out)} | CRS: {back.crs == gdf.crs} | "
              f"total area: {abs(back.geometry.area.sum() - out['area_m2'].sum()) < 1.0}")
    return out.set_index("id")


def gpkg_to_gdf(gpkg_path, layer: str = "change_features") -> gpd.GeoDataFrame:
    gpkg_path = Path(gpkg_path)
    if not gpkg_path.exists():
        raise FileNotFoundError(f"GeoPackage not found: {gpkg_path}")
    return gpd.read_file(gpkg_path, layer=layer).set_index("id")

# ======================= step 6: map =======================
def build_map(gdf, aoi_path, html_path):
    keep = [c for c in ["change_type", "area_ha", "conf_mean", "confidence_class"] if c in gdf.columns]
    g = gdf[keep + [gdf.geometry.name]].to_crs(4326)
    minx, miny, maxx, maxy = g.total_bounds

    esri = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
    m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=12,
                   tiles=esri, attr="Esri World Imagery")

    if aoi_path and Path(aoi_path).exists():                           # AOI: outline only, no fill
        aoi = gpd.read_file(aoi_path)
        if aoi.crs is None:
            aoi = aoi.set_crs(4326)
        aoi = aoi.to_crs(4326)[["geometry"]]
        aoi.explore(m=m, name="AOI", style_kwds={"fill": False, "color": "#00e5ff", "weight": 3})
        minx, miny, maxx, maxy = aoi.total_bounds
    else:
        print("AOI file not found: map drawn without the AOI outline.")

    for kind, color in [("loss", "#ff2d2d"), ("gain", "#2dff4f")]:    # one layer per type, toggled in the layer control
        sub = g[g["change_type"] == kind]
        if len(sub):
            sub.explore(m=m, color=color, name=kind.capitalize(),
                        style_kwds={"weight": 1, "fillOpacity": 0.55}, tooltip=keep, popup=True)

    folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[miny, minx], [maxy, maxx]])
    ensure_dir(Path(html_path).parent)
    m.save(str(html_path))
    print(f"Map saved: {html_path}")
    return m


# ======================= the whole pipeline =======================
def run_pipeline(*, run_unzip, zip_path, unzip_to, folder_date_1, folder_date_2, aoi_geojson_path,
                 band_blue, band_green, band_red, boa_offset,
                 detection_method, K, min_delta, min_pixels,
                 output_dir, change_map_name, change_binary_name, sqlite_name, table_name, map_html_name,
                 save_stacks, save_rgb_previews, reload_check, show_plots):
    folders = [folder_date_1, folder_date_2]
    bands = (band_blue, band_green, band_red)
    out = ensure_dir(output_dir)

    if run_unzip:
        print("\n=== 1. Unzip ===")
        unzip_data(zip_path, unzip_to)

    print("\n=== 2. Check inputs ===")
    check_inputs(folders, bands, show=show_plots)
    if save_stacks:
        for f in folders:
            stack_bands(f, bands, out / "stacks")
    if save_rgb_previews:
        rgbs = [rgb_preview(f, bands, out / "previews" / f"{Path(f).name}.png") for f in folders]
        if show_plots:
            fig, axes = plt.subplots(1, 2, figsize=(16, 8))
            for ax, rgb, f in zip(axes, rgbs, folders):
                ax.imshow(rgb); ax.set_title(Path(f).name); ax.axis("off")
            plt.tight_layout(); plt.show()

    print("\n=== 3. Detect changes ===")
    change, proba = detect_changes(folder_date_1, folder_date_2, bands, boa_offset,
                                   detection_method, K, min_delta,
                                   out, change_map_name, change_binary_name, show_plots)

    print("\n=== 4. Extract polygons ===")
    gdf = extract_polygons(folder_date_1, band_blue, change, min_pixels, proba, show_plots)

    print("\n=== 5. Save to SQLite ===")
    gdf_to_gpkg(gdf, out / sqlite_name, layer=table_name, reload=reload_check)

    print("\n=== 6. Map ===")
    m = build_map(gdf, aoi_geojson_path, out / map_html_name)

    print("\n✅ Pipeline finished.")
    return {"change": change, "proba": proba, "gdf": gdf, "map": m}

In [ ]:
#@title 2. Pipeline parameters

#@markdown ### Inputs
run_unzip = True #@param {type:"boolean"}
zip_path = "/content/solafune_task.zip" #@param {type:"string"}
unzip_to = "/content" #@param {type:"string"}
folder_date_1 = "/content/solafune_task/data/sentinel2_20230812" #@param {type:"string"}
folder_date_2 = "/content/solafune_task/data/sentinel2_20230902" #@param {type:"string"}
aoi_geojson_path = "/content/solafune_task/aoi.geojson" #@param {type:"string"}

#@markdown ### Bands (file names without .tif, in Blue, Green, Red order)
band_blue = "B02" #@param {type:"string"}
band_green = "B03" #@param {type:"string"}
band_red = "B04" #@param {type:"string"}
boa_offset = 0 #@param {type:"integer"}
#@markdown Set `boa_offset` to 1000 if the files carry the +1000 offset (L2A baseline 04.00 or later).

#@markdown ### Detection
detection_method = "MAD" #@param ["MAD", "kmeans (experimental)"]
K = 4 #@param {type:"slider", min:1, max:6, step:0.5}
min_delta = 0.1 #@param {type:"number"}
#@markdown `K` is used by MAD only. `min_delta` is used by Kmeans (experimental) only.

#@markdown ### Polygons
min_pixels = 50 #@param {type:"integer"}
#@markdown 50 pixels = 0.5 ha at 10 m resolution.

#@markdown ### Outputs
output_dir = "/content/solafune_task/data/processed" #@param {type:"string"}
change_map_name = "change_map.tif" #@param {type:"string"}
change_binary_name = "change_binary.tif" #@param {type:"string"}
sqlite_name = "feature_change_extraction.gpkg" #@param {type:"string"}
table_name = "change_polygons" #@param {type:"string"}
map_html_name = "change_map.html" #@param {type:"string"}

#@markdown ### Switches
save_stacks = True #@param {type:"boolean"}
save_rgb_previews = True #@param {type:"boolean"}
reload_check = True #@param {type:"boolean"}
show_plots = True #@param {type:"boolean"}

In [ ]:
#@title Run Pipeline
import ipywidgets as widgets
from IPython.display import display


# Define the function you want the button to run
def on_button_clicked(b):
  print("Button clicked! Executing code...")
  result = run_pipeline(
      run_unzip=run_unzip, zip_path=zip_path, unzip_to=unzip_to,
      folder_date_1=folder_date_1, folder_date_2=folder_date_2, aoi_geojson_path=aoi_geojson_path,
      band_blue=band_blue, band_green=band_green, band_red=band_red, boa_offset=boa_offset,
      detection_method=detection_method, K=K, min_delta=min_delta, min_pixels=min_pixels,
      output_dir=output_dir, change_map_name=change_map_name, change_binary_name=change_binary_name,
      sqlite_name=sqlite_name, table_name=table_name, map_html_name=map_html_name,
      save_stacks=save_stacks, save_rgb_previews=save_rgb_previews,
      reload_check=reload_check, show_plots=show_plots,
  )

  display(result["map"])     # the last line of the cell displays the interactive map


button = widgets.Button(description="Run Cell")
button.on_click(on_button_clicked)
display(button)

## Appendix A

Query through geopackage file

In [ ]:
import geopandas as gpd

gpkg_path = "/content/solafune_task/data/processed/feature_change_extraction.gpkg"
gdf = gpd.read_file(gpkg_path)
gdf.head(5)

,id,class_id,change_type,area_ha,area_m2,confidence,geometry
0,1,2,gain,2.13,21300.0,NaN,"POLYGON ((369260 8653490, 369260 8653480, 3692..."
1,2,2,gain,0.55,5500.0,NaN,"POLYGON ((375700 8653170, 375700 8653160, 3756..."
2,3,2,gain,0.58,5800.0,NaN,"POLYGON ((375900 8653070, 375900 8653060, 3759..."
3,4,2,gain,3.16,31600.0,NaN,"POLYGON ((369300 8652840, 369300 8652830, 3692..."
4,5,1,loss,0.90,9000.0,NaN,"POLYGON ((379500 8649730, 379500 8649720, 3794..."


In [ ]:
!sudo apt install libsqlite3-mod-spatialite

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libsqlite3-mod-spatialite
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,927 kB of archives.
After this operation, 8,029 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libsqlite3-mod-spatialite amd64 5.1.0-3build1 [1,927 kB]
Fetched 1,927 kB in 1s (1,707 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libsqlite3-mod-

In [ ]:
# Install mod_spatialite dependencies in ubuntu env or in Google Colab
!apt-get install -y libspatialite-dev spatialite-bin > /dev/null

import sqlite3
import pandas as pd
import os

# Establish connection to the GPKG file as a SQLite database
conn = sqlite3.connect(output_dir + '/' + sqlite_name)

# Enable extension loading
conn.enable_load_extension(True)

# Load mod_spatialite. In Debian/Ubuntu environments, it is typically loaded via its full library path
# or dynamically mapped through standard system paths as 'mod_spatialite'
try:
    conn.load_extension("mod_spatialite")
except sqlite3.OperationalError:
    # Fallback to absolute system path if standard loading is blocked
    spatialite_path = "/usr/lib/x86_64-linux-gnu/mod_spatialite.so"
    if os.path.exists(spatialite_path):
        conn.load_extension(spatialite_path)
    else:
        raise

# Enable GPKG mode so SpatiaLite spatial functions accept GPKG blobs directly
conn.execute("SELECT EnableGpkgMode();")

# Query using spatial functions
query = f"""
    SELECT fid, change_type, ST_Area(geom) AS area_calc_m2, area_m2
    FROM {table_name}
    LIMIT 5
"""

df_query = pd.read_sql_query(query, conn)

# Close connection and display results
conn.close()
display(df_query)

,fid,change_type,area_calc_m2,area_m2
0,1,gain,21300.0,21300.0
1,2,gain,5500.0,5500.0
2,3,gain,5800.0,5800.0
3,4,gain,31600.0,31600.0
4,5,loss,9000.0,9000.0
